# 🩺 Disease Prediction: Model Analysis & Visualizations

This notebook provides a comprehensive evaluation of the Random Forest model used in the Medical Broker app with explicit **Label Encoding** for diseases.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import joblib
import os

# Plotting settings
%matplotlib inline
sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

## 1. Load Data

In [ ]:
data_path = "Main_Training_2026.csv"

if not os.path.exists(data_path):
    print(f"❌ Error: {data_path} not found.")
else:
    df = pd.read_csv(data_path)
    for col in ["prognosis", "disease"]:
        if col in df.columns:
            df.rename(columns={col: "Disease"}, inplace=True)
            
    print(f"✅ Data loaded: {df.shape[0]} rows")

## 2. Preprocessing & Label Encoding

In [ ]:
drop_cols = [c for c in ["Disease", "medicine"] if c in df.columns]
X = df.drop(columns=drop_cols).select_dtypes(include=[np.number])
y_raw = df["Disease"].str.strip()

# Initialize and fit LabelEncoder
le = LabelEncoder()
y = le.fit_transform(y_raw)

# Display class mapping
mapping = dict(zip(le.classes_, range(len(le.classes_))))
print("Mapping samples (first 5):")
for k in list(mapping.keys())[:5]:
    print(f"  {k} -> {mapping[k]}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## 3. Model Training

In [ ]:
clf = RandomForestClassifier(n_estimators=150, random_state=42, class_weight="balanced", n_jobs=-1)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"⭐ Model Accuracy: {acc*100:.2f}%")

## 4. Visualizations

In [ ]:
# A. Feature Importance
importances = clf.feature_importances_
indices = np.argsort(importances)[::-1]
top_k = 20
sns.barplot(x=importances[indices[:top_k]], y=X.columns[indices[:top_k]], palette="viridis")
plt.title(f"Top {top_k} Important Symptoms")
plt.show()

# B. Confusion Matrix (Decoding labels for axis)
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(15, 12))
sns.heatmap(cm, annot=False, cmap="Blues", xticklabels=le.classes_, yticklabels=le.classes_)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks(rotation=90)
plt.show()

## 5. Detailed Report

In [ ]:
print(classification_report(y_test, y_pred, target_names=le.classes_))

## 6. Export

In [ ]:
joblib.dump((clf, X.columns.values, acc, le), "model_cache.pkl")
print("✅ Model and LabelEncoder saved to model_cache.pkl")